# 01 — Environment Setup & Baseline Evaluation

**Phase 1 / Month 1** · MSc Thesis.

University of West Attica (UniWA / AIDL)  

Supervisor: Dr. Panagiotis Kasnesis 

Student: Antonios Bastoulis

---

This notebook establishes the CityLearn v2 environment and two classical baselines that anchor the thesis evaluation. Everything here runs locally on CPU at full-year scale (8 759 hourly timesteps).

**Evaluation standard — 2022 CityLearn Challenge scoring** 
(Appendix A of the challenge paper). All metrics are normalised against the no-battery-control baseline → **1.0 = no improvement, < 1.0 = better**.

| Symbol | Meaning | CityLearn v2 KPI name |
|---|---|---|
| **C** | Cost ratio | `district_cost_ratio_to_baseline_total_ratio` |
| **G** | Carbon ratio | `district_emissions_ratio_to_baseline_total_ratio` |
| **R** | Ramping ratio | `district_energy_grid_shape_quality_ramping_average_to_baseline_ratio` |
| **1−L** | Load-factor penalty ratio | `district_energy_grid_shape_quality_load_factor_penalty_daily_average_to_baseline_ratio` |
| **Phase I** | **(C + G) / 2** —  | — |
| **Combined** | **(C + G + D) / 3**, D = (R + 1−L) / 2 | — |

## 0. Config

All experiment constants live in this single cell — the seed, dataset path, building split (train vs held-out), episode bounds, and the SAC training budget. Re-runs of the notebook should never need edits anywhere else.

The defaults are kept in sync with `src/env.py` (the project-wide source of truth used by notebooks 02–07). The duplicated `MERLINReward` definition in § 2 is pedagogical only — it exists so this notebook reads as a self-contained explanation of the reward function.

In [ ]:
from pathlib import Path

# ── Reproducibility ──────────────────────────────────────────────────────
SEED: int = 42

# ── Dataset ──────────────────────────────────────────────────────────────
DATASET_ROOT = Path("../data/citylearn_datasets/citylearn_challenge_2022_phase_all")

BUILDINGS:        list = [0, 1, 2, 3, 4, 5]    # training buildings
UNSEEN_BUILDINGS: list = [6, 7, 8, 9, 10, 11]  # held-out generalisation set

SIM_START: int = 0
# simulation_end_time_step is INCLUSIVE in CityLearn 2.6 — end=8759 ⇒ 8760
# hourly steps (indices 0..8759, the full calendar year). The previous value
# (8758) was one step short of a year. Aligned with src/env.py:SIM_END so
# downstream SFT / RL all see the same episode length.
SIM_END:   int = 8759   # full year — 8 760 hourly steps

# ── Observations ─────────────────────────────────────────────────────────
# 9 real-time signals only.
#
# The CityLearn 2022 schema also exposes oracle short-horizon forecast columns
# (price +6 h / +12 h, solar irradiance +6 h). We deliberately do NOT use them:
# those values are perfect look-ahead reads from the simulation tape, not a
# signal an agent could realistically obtain in deployment. Including them
# would let any policy — SAC or LLM — "cheat" against the evaluation
# distribution and inflate KPIs. The agent must instead anticipate future
# conditions from real-time state alone (hour, day_type, present price/solar
# trend), which is the regime we care about for the thesis research questions.
OBSERVATIONS: list = [
    "month", "hour", "day_type",
    "electrical_storage_soc",
    "net_electricity_consumption",
    "non_shiftable_load",
    "solar_generation",
    "electricity_pricing",
    "carbon_intensity",
]

ACTIVE_ACTIONS: list = ["electrical_storage"]

# ── Training ─────────────────────────────────────────────────────────────
SAC_EPISODES: int = 15   

# ── Output ───────────────────────────────────────────────────────────────
ARTIFACTS = Path("artifacts").resolve()
ARTIFACTS.mkdir(exist_ok=True)

print(f"Observations : {len(OBSERVATIONS)}  (real-time only — no oracle forecasts)")
print(f"Buildings    : train={BUILDINGS}  unseen={UNSEEN_BUILDINGS}")
print(f"Episode      : t={SIM_START}..{SIM_END}  ({SIM_END - SIM_START + 1} steps)")

## 1. Imports

CityLearn v2 plus the two baseline agents shipped with it: `BasicBatteryRBC` (rule-based) and `SAC` (Soft Actor-Critic). Nothing here imports from `src/` — Phase 1 must read as a self-contained reference for the environment, baselines, and KPI scoring. Later notebooks (02 onward) reuse the same building blocks via `src.env`, `src.eval`, etc.

In [2]:
import json
import time
import warnings
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import citylearn
from citylearn.citylearn import CityLearnEnv
from citylearn.reward_function import RewardFunction
from citylearn.agents.rbc import BasicBatteryRBC
from citylearn.agents.sac import SAC

warnings.filterwarnings("ignore")
np.random.seed(SEED)

print(f"CityLearn {citylearn.__version__}")

CityLearn 2.6.0b2


## 2. Environment

### Observations (9 variables — real-time only)

| Variable | Type | Why |
|---|---|---|
| `month`, `hour`, `day_type` | Temporal | Seasonal and diurnal charge patterns; also the agent's only basis for anticipating upcoming price / solar windows |
| `electrical_storage_soc` | Battery | Avoid over/under charge |
| `net_electricity_consumption` | Grid | Direct reward signal (positive = import, negative = export) |
| `non_shiftable_load` | Demand | Anticipate solar headroom |
| `solar_generation` | PV | Decide when to charge from local generation |
| `electricity_pricing` | Tariff | Price-aware dispatch |
| `carbon_intensity` | Grid carbon | Carbon-aware dispatch |

**On the omitted forecast variables.** The CityLearn 2022 dataset also exposes oracle forecast columns — `electricity_pricing_predicted_1/2` (price +6 h, +12 h) and `diffuse/direct_solar_irradiance_predicted_1` (solar +6 h). Nweye et al. (2024, MERLIN) include them and show they boost SAC performance on this dataset. We deliberately exclude them in this thesis. Those values are perfect look-ahead reads from the simulation tape, not a signal an agent could realistically obtain in a real building deployment, and feeding them to the policy effectively lets it peek at the test distribution. Since one of the thesis research questions is how the SLM agent generalises to *unseen* buildings and weather, we want every policy under comparison (RBC, SAC, LLM-as-policy, fine-tuned SLM) to plan ahead from real-time state alone. This makes our baselines weaker than MERLIN's published numbers by design, but the comparison stays apples-to-apples and the deployment story stays honest.

### Action space

`electrical_storage` — continuous in [−1, +1]. Positive = charge, negative = discharge. Only the battery is controllable; the 2022 buildings have no active HVAC loads.

### Reward function

**`MERLINReward`** (Nweye et al., 2024):

```
reward = −(1 + sign(net) · SoC) · |net_consumption|
```

The SoC term amplifies the penalty when the battery is full during a grid-import step (missed discharge opportunity) and softens it when the battery is empty. Uses raw kWh values — no dataset-specific normalisation constants required. Selected because it is simple, published, and dataset-agnostic. It is the single reward function for the project, applied by `make_env()`.

> Nweye, K., Sankaranarayanan, S., & Nagy, G. (2024). MERLIN: Multi-agent offline and transfer learning for occupant-centric operation of grid-interactive communities. *Applied Energy*, 358, 121958. https://doi.org/10.1016/j.apenergy.2023.121958

In [ ]:
import json as _json

SCHEMA_FILE = DATASET_ROOT / "schema.json"
RENDER_DIR  = (ARTIFACTS / "SimulationData").resolve()
RENDER_DIR.mkdir(parents=True, exist_ok=True)


def load_schema() -> dict:
    """Load schema and patch root_directory to an absolute path."""
    with open(SCHEMA_FILE) as f:
        schema = _json.load(f)
    schema["root_directory"] = str(DATASET_ROOT.resolve())
    return schema


# ── Reward function ───────────────────────────────────────────────────────
# Pedagogical note: this class is also exported from `src.env` for the rest
# of the project (notebooks 02–07 import it). It is duplicated here only so
# § 2 reads as a self-contained explanation of the reward function. Keep it
# BYTE-FOR-BYTE in sync with src/env.py.

class MERLINReward(RewardFunction):
    """SoC-aware net-consumption reward (Nweye et al., 2024).

    reward = −(1 + sign(net) · SoC) · |net_consumption|

    Uses raw kWh values — no dataset-specific normalisation required.
    Grid-searched optimal parameters (Table 3): w1=1, w2=0, e1=1, e2=1.

    Reference: Nweye et al. (2024). Applied Energy, 358, 121958.
    https://doi.org/10.1016/j.apenergy.2023.121958
    """

    def __init__(self, env_metadata: dict, w1: float = 1.0, w2: float = 0.0,
                 e1: float = 1.0, e2: float = 1.0):
        super().__init__(env_metadata)
        self.w1, self.w2, self.e1, self.e2 = w1, w2, e1, e2

    def calculate(self, observations: list[dict]) -> list[float]:
        rewards = []
        for o in observations:
            net    = o.get("net_electricity_consumption", 0.0)
            carbon = o.get("carbon_intensity", 0.0) * abs(net)
            soc    = o.get("electrical_storage_soc", 0.0)
            signal = self.w1 * (abs(net) ** self.e1) + self.w2 * (abs(carbon) ** self.e2)
            rewards.append(float(-(1.0 + np.sign(net) * soc) * signal))
        return [float(sum(rewards))] if self.central_agent else rewards


# ── Environment factory ───────────────────────────────────────────────────

def make_env(buildings: list = None, session_name: str = None,
             render_mode: str = "end") -> CityLearnEnv:
    """Build a CityLearnEnv with MERLINReward and the 9-variable observation set.

    Args:
        buildings:    Building indices. Defaults to BUILDINGS.
        session_name: Enables render output to RENDER_DIR/<session_name>.
        render_mode:  'end' (buffer per episode) or 'during' (live writes).
    """
    kwargs = dict(
        schema=load_schema(),
        buildings=buildings if buildings is not None else BUILDINGS,
        central_agent=False,
        active_actions=ACTIVE_ACTIONS,
        active_observations=OBSERVATIONS,
        random_seed=SEED,
        simulation_start_time_step=SIM_START,
        simulation_end_time_step=SIM_END,
    )
    if session_name:
        kwargs.update(render_mode=render_mode,
                      render_directory=str(RENDER_DIR),
                      render_session_name=session_name)

    env = CityLearnEnv(**kwargs)
    env.reward_function = MERLINReward(env.get_metadata())
    return env


# ── Sanity check ─────────────────────────────────────────────────────────
_env = make_env()
_obs, _ = _env.reset()
print(f"buildings      : {len(_obs)}")
print(f"obs / building : {len(_obs[0])}")
print(f"action dims    : {len(_env.action_space[0].low)}")
print(f"steps / episode: {_env.time_steps}")
print(f"reward         : {type(_env.reward_function).__name__}")

## 3. RBC Baseline

`BasicBatteryRBC` is a fixed, zero-learning rule that requires no training. It serves as a lower bound — any learning agent that cannot beat a static rule has a broken learning signal.

**Rule:** charge at 11 % of capacity per hour during 06:00–14:00 (solar window), discharge at 6.7 % at all other hours. No price or carbon awareness.

In [4]:
env_rbc   = make_env(session_name="rbc_baseline")
agent_rbc = BasicBatteryRBC(env=env_rbc)

t0 = time.time()
agent_rbc.learn(episodes=1)
print(f"RBC done in {time.time() - t0:.1f}s")

RBC done in 19.3s


## 4. SAC Baseline

SAC is the learned upper bound. One independent policy network per building (`central_agent=False`), matching the decentralised architecture planned for the SLM agents. The final episode runs deterministically (`deterministic_finish=True`) for clean KPI measurement.

In [5]:
env_sac   = make_env(session_name="sac_training")
agent_sac = SAC(env=env_sac, seed=SEED)

print(f"Training SAC — {SAC_EPISODES} episodes (last deterministic)")
t0 = time.time()
agent_sac.learn(episodes=SAC_EPISODES, deterministic_finish=True)
print(f"SAC done in {time.time() - t0:.1f}s")

Training SAC — 15 episodes (last deterministic)
SAC done in 8479.6s


## 5. Evaluation

All metrics use `env.evaluate_v2()` and are normalised against the no-battery-control baseline (1.0 = no improvement, lower is better).

**Challenge KPIs (C, G, R, 1−L)** map 1:1 to the 2022 competition scoring — the challenge organisers used CityLearn's own evaluation functions.

**ZNE ratio** (Nweye et al. 2024): total solar generation ÷ total grid import. ≥ 1.0 = Zero Net Energy achieved. Values below 1.0 are typical for residential PV + battery: overnight charging increases grid imports beyond what daytime solar offsets.

In [ ]:
CHALLENGE_KPIS = {
    "C  — cost":        "district_cost_ratio_to_baseline_total_ratio",
    "G  — carbon":      "district_emissions_ratio_to_baseline_total_ratio",
    "R  — ramping":     "district_energy_grid_shape_quality_ramping_average_to_baseline_ratio",
    "1L — load factor": "district_energy_grid_shape_quality_load_factor_penalty_daily_average_to_baseline_ratio",
}


def district_kpis(env_obj: CityLearnEnv) -> pd.Series:
    """Return district-level KPIs from evaluate_v2() as a Series."""
    df   = env_obj.evaluate_v2()
    mask = df["level"].astype(str).str.lower() == "district"
    return df[mask].set_index("cost_function")["value"].astype(float)


def challenge_score(env_obj: CityLearnEnv, label: str) -> dict:
    """Compute the 2022 Challenge scores from evaluate_v2().

    Phase I  = (C + G) / 2        — primary thesis metric
    D        = (R + (1-L)) / 2    — grid quality
    Combined = (C + G + D) / 3   — full score
    """
    kpis = district_kpis(env_obj)
    C   = float(kpis[CHALLENGE_KPIS["C  — cost"]])
    G   = float(kpis[CHALLENGE_KPIS["G  — carbon"]])
    R   = float(kpis[CHALLENGE_KPIS["R  — ramping"]])
    oml = float(kpis[CHALLENGE_KPIS["1L — load factor"]])
    D   = (R + oml) / 2
    return {
        "agent":          label,
        "C  — cost":      round(C,   4),
        "G  — carbon":    round(G,   4),
        "R  — ramping":   round(R,   4),
        "1L — load factor": round(oml, 4),
        "Phase I (C+G)/2":  round((C + G) / 2, 4),
        "Combined (C+G+D)/3": round((C + G + D) / 3, 4),
    }


def zne_metric(env_obj: CityLearnEnv, label: str) -> dict:
    """ZNE ratio and self-consumption ratio (Nweye et al., 2024)."""
    d = district_kpis(env_obj)
    solar = float(d.get("district_solar_self_consumption_total_generation_kwh", 0.0))
    imp   = float(d.get("district_energy_grid_total_import_control_kwh",        1.0))
    sc    = float(d.get("district_solar_self_consumption_ratio_self_consumption_ratio", float("nan")))
    return {
        "agent":                      label,
        "solar generation (kWh)":     round(solar,  1),
        "grid import (kWh)":          round(imp,    1),
        "ZNE ratio (solar / import)": round(solar / max(imp, 1e-6), 4),
        "ZNE achieved (≥ 1.0)":       solar >= imp,
        "self-consumption ratio":     round(sc, 4),
    }


# ── Compute all scores ───────────────────────────────────────────────────
scores = pd.DataFrame([
    challenge_score(env_rbc, "RBC"),
    challenge_score(env_sac, "SAC"),
]).set_index("agent")

zne = pd.DataFrame([
    zne_metric(env_rbc, "RBC"),
    zne_metric(env_sac, "SAC"),
]).set_index("agent")

# ── Challenge KPI table ──────────────────────────────────────────────────
print("Challenge scores — lower is better; 1.0 = no-battery baseline")
display(scores)


# ── ZNE table ────────────────────────────────────────────────────────────
print("\nZNE metric — Nweye et al. (2024, MERLIN)")
display(zne)

## 6. Generalisation — Held-Out Buildings

The SLM agents will be tested on buildings they have not seen during training. This section measures the transfer gap using the classical baselines on the 6 held-out buildings (indices 6–11).

- **RBC** is stateless → generalises by construction. Its scores on unseen buildings measure environment difficulty, not learned behaviour.
- **SAC** runs inference-only (no weight updates) on the unseen district, using the same policy networks trained on buildings 0–5. Any performance drop is cross-building policy transfer degradation.

The **generalisation gap** (SAC unseen − SAC train on Phase I) directly quantifies how hard the SLM transfer problem is.

In [ ]:
# ── RBC on unseen buildings ───────────────────────────────────────────────
env_gen_rbc   = make_env(buildings=UNSEEN_BUILDINGS, session_name="rbc_generalization")
agent_gen_rbc = BasicBatteryRBC(env=env_gen_rbc)
t0 = time.time()
agent_gen_rbc.learn(episodes=1)
print(f"RBC unseen done in {time.time() - t0:.1f}s")

# SAC has one policy per building (central_agent=False); since len(UNSEEN_BUILDINGS) == len(BUILDINGS),
# policy i (trained on building i) is applied positionally to unseen building i — deliberate transfer test.
env_gen_sac = make_env(buildings=UNSEEN_BUILDINGS, session_name="sac_generalization")
obs, _ = env_gen_sac.reset()
t0 = time.time()
done = False
while not done:
    actions = agent_sac.predict(obs, deterministic=True)
    obs, _, terminated, truncated, _ = env_gen_sac.step(actions)
    done = bool(terminated or truncated)
print(f"SAC unseen done in {time.time() - t0:.1f}s")

In [ ]:
# ── Challenge scores: train vs unseen ────────────────────────────────────
gen_scores = pd.DataFrame([
    challenge_score(env_rbc,     "RBC  (train)"),
    challenge_score(env_gen_rbc, "RBC  (unseen)"),
    challenge_score(env_sac,     "SAC  (train)"),
    challenge_score(env_gen_sac, "SAC  (unseen)"),
]).set_index("agent")

print("Generalisation — challenge scores (lower is better)")
display(gen_scores)

# ── ZNE: train vs unseen ─────────────────────────────────────────────────
gen_zne = pd.DataFrame([
    zne_metric(env_rbc,     "RBC  (train)"),
    zne_metric(env_gen_rbc, "RBC  (unseen)"),
    zne_metric(env_sac,     "SAC  (train)"),
    zne_metric(env_gen_sac, "SAC  (unseen)"),
]).set_index("agent")

print("\nZNE — train vs unseen buildings")
display(gen_zne[["ZNE ratio (solar / import)", "self-consumption ratio"]])

# ── Generalisation gap ────────────────────────────────────────────────────
rbc_gap = gen_scores.loc["RBC  (unseen)", "Phase I (C+G)/2"] - gen_scores.loc["RBC  (train)",  "Phase I (C+G)/2"]
sac_gap = gen_scores.loc["SAC  (unseen)", "Phase I (C+G)/2"] - gen_scores.loc["SAC  (train)",  "Phase I (C+G)/2"]
print(f"\nPhase I generalisation gap (unseen − train):")
print(f"  RBC : {rbc_gap:+.4f}  (should be ≈0; measures environment difficulty)")
print(f"  SAC : {sac_gap:+.4f}  (positive = policy degrades on unseen buildings)")
if abs(sac_gap) > abs(rbc_gap) + 0.01:
    print("  → SAC shows clear transfer degradation beyond environment difficulty.")
else:
    print("  → SAC generalises comparably to RBC.")